In [ ]:
import torch as tr
import torch.nn as nn
import torch.autograd as autograd

import import_ipynb
from common import assert_eq, T # type: ignore

$$ f=x+1 $$
$$ \frac{\partial f}{\partial x} = 1 $$
$$ \diamond \diamond \diamond $$
$$ \text{Let } F=3+f $$
$$ \frac{\partial F}{\partial f} = 1 $$
$$ \frac{\partial F}{\partial x} = \frac{\partial F}{\partial f} \cdot \frac{\partial f}{\partial x} = 1$$

In [ ]:
class Add1Function(autograd.Function):
    @staticmethod
    def forward(ctx, x):
        return x + 1

    @staticmethod
    def backward(ctx, dF_df):
        # For this example only
        # assert_eq(dF_df.shape, (1,))
        # assert_eq(dF_df.item(), 1.0)

        df_dx = 1
        dF_dx = dF_df * df_dx

        return (dF_dx, )
   

class Add1(nn.Module):
    def forward(self, x):
        return Add1Function.apply(x)
    
    # `Module.backward()` is intentionally disabled.
    # Autograd never calls `backward` on modules.
    # Only Functions participate in the computational graph.
    def backward(self, *_):
        assert(False)
   

def _test_Add1():
    x = tr.tensor(4, dtype=tr.float32, requires_grad=True)
    F = 3 + Add1()(x)     
    F.backward()     

    assert_eq(F.item(), 3.0 + (4.0 + 1))
    assert_eq(x.grad.item(), 1.0)


if __name__ == "__main__":
    _test_Add1()

$$ f=2x $$
$$ \frac{\partial f}{\partial x} = 2 $$
$$ \diamond \diamond \diamond $$
$$ \text{Let } F=1+3f $$
$$ \frac{\partial F}{\partial f} = 3 $$
$$ \frac{\partial F}{\partial x} = \frac{\partial F}{\partial f} \cdot \frac{\partial f}{\partial x} $$

In [ ]:
class Mul2Function(autograd.Function):
    @staticmethod
    def forward(ctx, x):
        return 2 * x

    @staticmethod
    def backward(ctx, dF_df):
        # For this example only
        # assert_eq(dF_df.shape, (1,))
        # assert_eq(dF_df.item(), 3.0)
        
        df_dx = 2
        dF_dx = dF_df * df_dx

        return (dF_dx, )
   

class Mul2(nn.Module):
    def forward(self, x):
        return Mul2Function.apply(x)
   
    # `Module.backward()` is intentionally disabled.
    # Autograd never calls `backward` on modules.
    # Only Functions participate in the computational graph.
    def backward(self, *_):
        assert(False)


def _test_Mul2():
    x = tr.tensor([4.0], requires_grad=True)
    F = 1 + 3 * Mul2()(x)     
    F.backward()     

    assert_eq(F.item(), 1.0 + 3.0 * (2.0 * 4.0))
    assert_eq(x.grad.item(), 3.0 * 2.0)


if __name__ == "__main__":
    _test_Mul2()

$$ f=xy $$
$$ \frac{\partial f}{\partial x} = y $$
$$ \frac{\partial f}{\partial y} = x $$
$$ \diamond \diamond \diamond $$
$$ \text{Let } F=2+5f $$
$$ \frac{\partial F}{\partial f} = 5 $$
$$ \frac{\partial F}{\partial x} = \frac{\partial F}{\partial f} \cdot \frac{\partial f}{\partial x} $$
$$ \frac{\partial F}{\partial y} = \frac{\partial F}{\partial f} \cdot \frac{\partial f}{\partial y} $$

In [ ]:
class MultiplyFunction(autograd.Function):
    @staticmethod
    def forward(ctx, x, y):
        ctx.save_for_backward(x, y)
        return x * y

    @staticmethod
    def backward(ctx, dF_df):
        # For this example only
        # assert_eq(dF_df.shape, (1,))
        # assert_eq(dF_df.item(), 5.0)

        (x, y) = ctx.saved_tensors

        df_dx = y
        dF_dx =  dF_df * df_dx

        df_dy = x
        dF_dy =  dF_df * df_dy

        return (dF_dx, dF_dy)
   

# `MultiplyFunction` implements the actual operator.
# `Multiply(Module)` is just a wrapper for nicer API.
class Multiply(nn.Module):
    def forward(self, x, y):
        return MultiplyFunction.apply(x, y)
   
    # `Module.backward()` is intentionally disabled.
    # Autograd never calls `backward` on modules.
    # Only Functions participate in the computational graph.
    def backward(self, *_):
        assert(False)


def _test_MultiplyFunction():
    x = tr.tensor([3.0], requires_grad=True)
    y = tr.tensor([4.0], requires_grad=True)

    F = 2.0 + 5.0 * Multiply()(x, y)     
    F.backward()        

    assert_eq(F.item(), 2.0 + 5.0 * (3.0 * 4.0))
    assert_eq(x.grad.item(), 4.0 * 5.0)
    assert_eq(y.grad.item(), 3.0 * 5.0)


if __name__ == "__main__":
    _test_MultiplyFunction()

$$ z=xW $$
$$ p=\frac{e^z}{1+e^z} $$
$$ L=-y\log p - (1-y)\log(1-p) $$

$$ \diamond \diamond \diamond $$

$$ \frac{\partial L}{\partial x} = \frac{\partial L}{\partial z} \cdot \frac{\partial z}{\partial x} = (p-y) \cdot W^T $$
$$ \frac{\partial L}{\partial W} = \frac{\partial L}{\partial z} \cdot \frac{\partial z}{\partial W} = x^T \cdot(p-y) $$

$$ \diamond \diamond \diamond $$

$$ \text{Vector inner product} $$
$$ \text{For } u \in R^n, v \in R^n, \langle u, v \rangle = \sum_i u_i v_i = u^T \cdot v $$

$$ \diamond \diamond \diamond $$

$$ \text{Frobenius inner product} $$
$$ \text{For } U \in R^{m \times n}, V \in R^{m \times n}, \langle U, V \rangle_{F} = \sum_{i,j} U_{ij} V_{ij} = \text{tr}(U^T V) $$

$$ \diamond \diamond \diamond $$

$$ \text{Trace properties} $$
$$ (1) \quad \text{tr}(AB) = \text{tr}(BA) $$
$$ (2) \quad \text{tr}(A^T B) = \text{tr}(AB^T) $$

$$ \diamond \diamond \diamond $$

$$ \text{Frobenius inner product properties} $$
$$ (3) \quad \Big\langle U, AV \Big\rangle_{F} = \text{tr}\Big(U^T (AV)\Big) = \text{tr}\Big((U^T A)V\Big) = \text{tr}\Big((A^T U)^T V\Big) = \Big\langle A^TU, V \Big\rangle_{F} $$
$$ (4) \quad\Big\langle U, VA \Big\rangle_{F} = \text{tr}\Big(U^T (VA)\Big) \overset{(1,2)}{=} \text{tr}\Big((A U^T) V\Big) = \Big\langle UA^T, V \Big\rangle_{F} $$

$$ \diamond \diamond \diamond $$

$$ dz = dx \, W $$
$$ dL = \Bigg\langle \frac{\partial L}{\partial z} , dz \Bigg\rangle_{F} = 
\Bigg\langle \frac{\partial L}{\partial z} , dx \, W \Bigg\rangle_{F} \overset{(4)}{=} 
\Bigg\langle \frac{\partial L}{\partial z} \, W^T , dx \Bigg\rangle_{F} $$

$$ \diamond \diamond \diamond $$

$$ dz = x \, dW $$
$$ dL = \Bigg\langle \frac{\partial L}{\partial z} , dz \Bigg\rangle_{F} = 
\Bigg\langle \frac{\partial L}{\partial z} , x \, dW \Bigg\rangle_{F} \overset{(3)}{=}
\Bigg\langle x^T \, \frac{\partial L}{\partial z} , dW \Bigg\rangle_{F} $$

In [ ]:
class ChainRuleFunction(autograd.Function):
    @staticmethod
    def forward(ctx, x, W, y):
      z = x @ W
      p = tr.exp(z) / (1 + tr.exp(z))
      L = -y * tr.log(p) - (1-y) * tr.log(1-p)

      ctx.save_for_backward(x, W, y, z, p)

      return L.mean()

    @staticmethod
    def backward(ctx, dF_df):
      (x, W, y, z, p) = ctx.saved_tensors

      dL_dz = (p - y) / x.shape[0]
      assert_eq(dL_dz.shape, z.shape)

      # RuntimeError: mat1 and mat2 shapes cannot be multiplied (5x2 and 3x2)
      # dL_dx = dL_dz @ W
      dL_dx = dL_dz @ W.t()
      assert_eq(dL_dx.shape, x.shape)

      # mat1 and mat2 shapes cannot be multiplied (5x2 and 5x3)
      # dL_dW = dL_dz @ x
      dL_dW = x.t() @ dL_dz
      assert_eq(dL_dW.shape, W.shape)

      return (dF_df * dL_dx, dF_df * dL_dW, None)


def _test_ChainRuleFunction():
  S = 5   # Samples
  FI = 3  # Features in
  FO = 2  # Features out

  x = tr.randn((S, FI), requires_grad=True, dtype=tr.float32)
  W = tr.randn((FI, FO), requires_grad=True, dtype=tr.float32)
  y = tr.randn((S, FO), dtype=tr.float32)
  
  L = ChainRuleFunction.apply(x, W, y)
  L.backward()


if __name__ == "__main__":
  _test_ChainRuleFunction()  
